# Notebook 5: Error Analysis and Consistency

**Project:** Evaluating the Functional Correctness and Consistency of AI-Generated Introductory Programming Solutions: Implications for Computing Education

**Author:** Dr. C. V. Krishnaveni

**Institution:** Department of Computer Science, SKR & SKR Government College for Women (Autonomous), Kadapa, Andhra Pradesh, India

---

## Objective

This notebook analyzes the functional correctness and consistency of AI-generated Python programs.

The analysis includes:

- Overall functional correctness
- Error analysis
- Solution-wise performance
- Problem-wise performance
- Consistency across multiple generations
- Summary statistics for research reporting

## Workflow

This notebook performs the following tasks:

1. Clone the GitHub repository.
2. Load evaluation results.
3. Compute descriptive statistics.
4. Analyze solution consistency.
5. Analyze error distribution.
6. Generate summary tables.
7. Export analysis results for publication.

In [28]:
# ============================================================
# Clone Repository
# ============================================================

!rm -rf AI-Code_Judgement

!git clone https://github.com/vkchennuru/AI-Code_Judgement.git

Cloning into 'AI-Code_Judgement'...
remote: Enumerating objects: 161, done.
remote: Counting objects: 100% (161/161), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 161 (delta 62), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (161/161), 395.34 KiB | 11.63 MiB/s, done.
Resolving deltas: 100% (62/62), done.


In [29]:
# ============================================================
# Import Libraries
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

In [30]:
# ============================================================
# Project Paths
# ============================================================

PROJECT_ROOT = Path("/content/AI-Code_Judgement")

RESULTS_DIR = PROJECT_ROOT / "results"

print("Project Root :", PROJECT_ROOT)
print("Results      :", RESULTS_DIR)

Project Root : /content/AI-Code_Judgement
Results      : /content/AI-Code_Judgement/results


## Load Functional Evaluation Results

In [31]:
evaluation_df = pd.read_csv(
    RESULTS_DIR / "evaluation_results.csv"
)

print("Rows :", len(evaluation_df))

display(evaluation_df.head())

Rows : 30


,experiment_id,problem_id,generation,program,syntax_correct,execution_success,passed_tests,total_tests,error_type,failed_test_case,test_category,expected_output,actual_output,stderr,execution_time_ms
0,PILOT_01,1,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,274.78
1,PILOT_01,1,B,solution_B.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,206.18
2,PILOT_01,1,C,solution_C.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,200.96
3,PILOT_01,2,A,solution_A.py,Yes,Yes,5,5,NaN,NaN,NaN,NaN,NaN,NaN,227.92
4,PILOT_01,2,B,solution_B.py,Yes,No,0,5,Runtime Error,1.0,Normal,78.54,NaN,"File ""/content/drive/MyDrive/AI-Code-Judgment-...",42.77


In [32]:
print(evaluation_df.columns.tolist())

['experiment_id', 'problem_id', 'generation', 'program', 'syntax_correct', 'execution_success', 'passed_tests', 'total_tests', 'error_type', 'failed_test_case', 'test_category', 'expected_output', 'actual_output', 'stderr', 'execution_time_ms']


## Dataset Overview

This section summarizes the evaluation dataset generated in Notebook 4.

In [33]:
# ============================================================
# Dataset Overview
# ============================================================

print("=" * 70)
print("Dataset Overview")
print("=" * 70)

print("Total Experiments :", len(evaluation_df))
print("Unique Problems   :", evaluation_df["problem_id"].nunique())
print("Generations       :", evaluation_df["generation"].unique())
print("Programs Tested   :", evaluation_df["program"].nunique())

Dataset Overview
Total Experiments : 30
Unique Problems   : 10
Generations       : ['A' 'B' 'C']
Programs Tested   : 3


## Overall Functional Correctness

This section summarizes the functional correctness of all AI-generated programs.

In [34]:
# ============================================================
# Functional Correctness
# ============================================================

overall = pd.DataFrame({

    "Metric":[
        "Syntax Correct",
        "Execution Successful"
    ],

    "Count":[

        evaluation_df["syntax_correct"].sum(),

        evaluation_df["execution_success"].sum()

    ]

})

display(overall)

,Metric,Count
0,Syntax Correct,YesYesYesYesYesYesYesYesYesYesYesYesYesYesYesY...
1,Execution Successful,YesYesYesYesNoYesYesYesYesYesYesYesYesYesNoNoN...


In [35]:
evaluation_df["accuracy"] = (
    evaluation_df["passed_tests"] /
    evaluation_df["total_tests"]
) * 100

print("Average Accuracy :",
      round(evaluation_df["accuracy"].mean(),2),
      "%")

display(
    evaluation_df[
        [
            "problem_id",
            "generation",
            "accuracy"
        ]
    ].head()
)

Average Accuracy : 56.67 %


,problem_id,generation,accuracy
0,1,A,100.0
1,1,B,100.0
2,1,C,100.0
3,2,A,100.0
4,2,B,0.0


In [36]:
# ============================================================
# Convert Yes/No columns to Boolean
# ============================================================

evaluation_df["execution_success"] = (
    evaluation_df["execution_success"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": True, "no": False})
)

evaluation_df["syntax_correct"] = (
    evaluation_df["syntax_correct"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map({"yes": True, "no": False})
)

print("Data types updated successfully.")

print(evaluation_df[["execution_success", "syntax_correct"]].dtypes)

Data types updated successfully.
execution_success    bool
syntax_correct       bool
dtype: object


## Generation-wise Performance

This section compares the functional correctness of different AI-generated solutions (Generation A, B, and C).

In [37]:
# ============================================================
# Generation-wise Accuracy
# ============================================================

generation_summary = (
    evaluation_df
    .groupby("generation")
    .agg(
        Programs=("program", "count"),
        Average_Accuracy=("accuracy", "mean"),
        Successful_Executions=("execution_success", "sum")
    )
    .round(2)
    .reset_index()
)

display(generation_summary)

,generation,Programs,Average_Accuracy,Successful_Executions
0,A,10,70.0,7
1,B,10,50.0,5
2,C,10,50.0,5


## Error Type Distribution

This section summarizes the types of errors observed during functional evaluation.

In [38]:
# ============================================================
# Error Type Distribution
# ============================================================

error_summary = (
    evaluation_df["error_type"]
    .fillna("No Error")
    .value_counts()
    .reset_index()
)

error_summary.columns = ["Error Type", "Frequency"]

display(error_summary)

,Error Type,Frequency
0,No Error,17
1,Runtime Error,13


## Problem-wise Functional Correctness

This section evaluates how well AI-generated programs performed on each benchmark problem.

In [39]:
# ============================================================
# Problem-wise Accuracy
# ============================================================

problem_summary = (
    evaluation_df
    .groupby("problem_id")
    .agg(
        Average_Accuracy=("accuracy", "mean"),
        Successful_Executions=("execution_success", "sum")
    )
    .round(2)
    .reset_index()
)

display(problem_summary)

,problem_id,Average_Accuracy,Successful_Executions
0,1,100.00,3
1,2,66.67,2
2,3,100.00,3
3,4,100.00,3
4,5,66.67,2
5,6,0.00,0
6,7,33.33,1
7,8,100.00,3
8,9,0.00,0
9,10,0.00,0


## Consistency Analysis

A problem is considered **consistent** if all three AI-generated solutions achieved the same functional outcome (all correct or all incorrect). Mixed outcomes indicate inconsistent behavior across generations.

In [40]:
# ============================================================
# Consistency Analysis
# ============================================================

consistency = (
    evaluation_df
    .groupby("problem_id")
    .agg(
        Min_Accuracy=("accuracy", "min"),
        Max_Accuracy=("accuracy", "max")
    )
)

consistency["Consistent"] = (
    consistency["Min_Accuracy"] ==
    consistency["Max_Accuracy"]
)

consistency = consistency.reset_index()

display(consistency)

,problem_id,Min_Accuracy,Max_Accuracy,Consistent
0,1,100.0,100.0,True
1,2,0.0,100.0,False
2,3,100.0,100.0,True
3,4,100.0,100.0,True
4,5,0.0,100.0,False
5,6,0.0,0.0,True
6,7,0.0,100.0,False
7,8,100.0,100.0,True
8,9,0.0,0.0,True
9,10,0.0,0.0,True


## Export Summary Tables

The generated summary tables are saved for reuse in the research paper.

In [41]:
# ============================================================
# Save Summary Tables
# ============================================================

generation_summary.to_csv(
    RESULTS_DIR / "generation_summary.csv",
    index=False
)

problem_summary.to_csv(
    RESULTS_DIR / "problem_summary.csv",
    index=False
)

error_summary.to_csv(
    RESULTS_DIR / "error_summary.csv",
    index=False
)

consistency.to_csv(
    RESULTS_DIR / "consistency_summary.csv",
    index=False
)

print("Summary tables saved successfully.")

Summary tables saved successfully.


## Reproducibility

This notebook analyzes the evaluation results generated in **Notebook 4: Functional Evaluation**.

To reproduce the analysis:

1. Clone the GitHub repository.
2. Execute Notebooks 1–4 in sequence.
3. Ensure that `results/evaluation_results.csv` has been generated.
4. Run this notebook from top to bottom without modifying intermediate files.

Outputs generated by this notebook include:

- `results/generation_summary.csv`
- `results/problem_summary.csv`
- `results/error_summary.csv`
- `results/consistency_summary.csv`

These summary tables serve as inputs for the visualization notebook and provide quantitative evidence for the experimental results reported in the ACM COMPUTE 2026 paper.

## Conclusion

This notebook analyzed the AI-generated Python solutions produced in Notebook 3 and evaluated in Notebook 4.

The analyses include:

- Overall functional correctness
- Generation-wise performance
- Problem-wise performance
- Error type distribution
- Consistency across multiple generations

The generated summary tables will be used for visualization and discussion in subsequent notebooks and for preparing the ACM COMPUTE 2026 paper.

In [42]:
print("=" * 70)
print("Notebook 5 Completed Successfully")
print("=" * 70)

print(f"Total Experiments : {len(evaluation_df)}")
print(f"Average Accuracy  : {evaluation_df['accuracy'].mean():.2f}%")
print(f"Problems Tested   : {evaluation_df['problem_id'].nunique()}")
print(f"Generations       : {evaluation_df['generation'].nunique()}")

print("\nGenerated Files:")
print("- generation_summary.csv")
print("- problem_summary.csv")
print("- error_summary.csv")
print("- consistency_summary.csv")

print("\nReady for Notebook 6: Visualization and Statistical Analysis")

Notebook 5 Completed Successfully
Total Experiments : 30
Average Accuracy  : 56.67%
Problems Tested   : 10
Generations       : 3

Generated Files:
- generation_summary.csv
- problem_summary.csv
- error_summary.csv
- consistency_summary.csv

Ready for Notebook 6: Visualization and Statistical Analysis
